# Superadditivity Analysis — PathMNIST, DomainNet-126, CIFAR-100

Reads experiment outputs from Google Drive and runs the **same analysis as
`scripts/run_analysis.py` and `scripts/run_visualization.py`** (this notebook
calls those functions; it does not re-implement any statistics):

- Factorial analysis (core_factorial): ANOVA, Freedman–Lane permutation test,
  seed-paired interaction test (I > 0), mixed-effects model
- Interaction surface I(α, β) (pout sweeps): per-point I with CI, surface fit
- Figures

All analyses use **drift orientation** (higher = more drift): CKA enters as
`1 - CKA`, MMD as is, so I > 0 means superadditive drift. Each analysis runs
for several metrics/scales (`ANALYSES` below) so the conclusion can be checked
on more than one scale.

## 1. Mount Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.isdir('/content/quantify_superadditivity'):
    !git clone https://github.com/angelafoua/quantify_superadditivity /content/quantify_superadditivity
%cd /content/quantify_superadditivity
!git checkout main && git pull origin main

In [ ]:
!pip install -q hydra-core==1.3.2 omegaconf==2.3.0 h5py networkx statsmodels seaborn tabulate tqdm scikit-learn pandas scipy
!pip install -q -e . --no-deps
!python scripts/validate_setup.py

In [ ]:
from pathlib import Path

# Point directly at the Drive folder — no download needed
DRIVE_OUT = Path('/content/drive/MyDrive/superadditivity_outputs')

# Also check Shared with me path in case the folder lives there
if not DRIVE_OUT.exists():
    DRIVE_OUT = Path('/content/drive/Shareddrives/superadditivity_outputs')
if not DRIVE_OUT.exists():
    # Shared-with-me folders appear under MyDrive once you add a shortcut,
    # or under this path on some Colab mounts:
    DRIVE_OUT = Path('/content/drive/MyDrive/superadditivity_outputs')
    print('NOTE: If the folder is shared with you (not owned), add a shortcut ')
    print('to your MyDrive first: Drive UI -> right-click folder -> "Add shortcut to Drive"')

assert DRIVE_OUT.exists(), f'{DRIVE_OUT} not found. Check the path or add a Drive shortcut.'

summaries = sorted(DRIVE_OUT.rglob('summary.json'))
print(f'Results root: {DRIVE_OUT}')
print(f'Found {len(summaries)} summary.json files')

## 2. Setup

In [ ]:
import logging
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

logging.basicConfig(level=logging.WARNING, format='%(levelname)s: %(message)s')

from superadditivity.analysis.aggregator import ResultAggregator
from superadditivity.analysis.drift_metrics import resolve_drift_metric, output_subdir
from scripts.run_analysis import run_factorial_analysis, run_surface_analysis
from scripts.run_visualization import plot_interaction_bar_figure, plot_trajectory_figure

# (metric, transform) pairs to analyse. 'cka' = 1 - client-level CKA between
# communities (primary); 'mmd' = paired MMD^2 between communities. The logit
# scale is the scale-robustness check for CKA.
ANALYSES = [('cka', 'none'), ('cka', 'logit'), ('mmd', 'none')]

# Analysis outputs are written locally, not into the shared Drive folder.
LOCAL_OUT = Path('/content/analysis_outputs')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
print('Modules loaded.')

## 3. Discover experiment folders

In [ ]:
def find_experiment_roots(root: Path) -> dict:
    """Map each experiment root folder to its type and dataset.

    The root is the first component of the run's experiment_name (e.g.
    'pout_sweep_pathmnist' for 'pout_sweep_pathmnist/pout0.07_pathmnist_iid').
    """
    roots = {}
    for summary in sorted(root.rglob('summary.json')):
        data = json.load(open(summary))
        top = data.get('experiment_name', '').split('/')[0]
        kind = ('core_factorial' if top.startswith('core_factorial')
                else 'pout_sweep' if top.startswith('pout_sweep') else None)
        if kind is None:
            continue
        entry = roots.setdefault(root / top, {'kind': kind, 'datasets': set(), 'n_runs': 0})
        entry['datasets'].add(data.get('dataset', 'unknown'))
        entry['n_runs'] += 1
    for r, e in roots.items():
        print(f"{e['kind']:15s} {r.name:35s} datasets={sorted(e['datasets'])} runs={e['n_runs']}")
    return roots

experiment_roots = find_experiment_roots(DRIVE_OUT)

## 4. Core Factorial Analysis

For each core_factorial folder and each (metric, transform) in `ANALYSES`, runs
`run_analysis.run_factorial_analysis`:
- Two-way ANOVA and Freedman–Lane permutation test of the interaction F
- **Primary:** interaction test paired by `run_seed` (one-sided t-test on per-seed I)
- Secondary: independent-cells bootstrap; mixed-effects model with crossed seed effects

Folders that are not a clean 2×2 (e.g. several α or p_out levels) raise an error
instead of being pooled.

In [ ]:
def run_factorial(root: Path, metric_name: str, transform: str) -> dict:
    metric = resolve_drift_metric(metric_name, transform=transform)
    out_dir = LOCAL_OUT / root.name / output_subdir(metric, 'analysis')
    try:
        run_factorial_analysis(str(root), str(out_dir), metric)
    except ValueError as exc:
        print(f'  {root.name} [{metric.key}]: skipped: {exc}')
        return {}
    result = json.load(open(out_dir / 'interaction_test.json'))
    result['report'] = str(out_dir / 'statistical_report.md')
    return result

In [ ]:
factorial_results = {}
for root, info in experiment_roots.items():
    if info['kind'] != 'core_factorial':
        continue
    for metric_name, transform in ANALYSES:
        key = (root.name, resolve_drift_metric(metric_name, transform=transform).key)
        factorial_results[key] = run_factorial(root, metric_name, transform)
        r = factorial_results[key]
        if r:
            print(f"{key[0]} [{key[1]}]: I = {r['I_observed']:.4g}, "
                  f"95% CI [{r['I_ci_lower']:.4g}, {r['I_ci_upper']:.4g}], "
                  f"p = {r['p_value']:.3g} ({r['n_units']} seeds)")

## 5. Pout Sweep — Interaction Surface I(α, β)

Runs `run_analysis.run_surface_analysis`: IID runs are the reference level
(α = ∞) with the largest p_out; points on the reference row/column (I = 0 by
construction) are excluded; each point gets a seed-paired CI.

In [ ]:
def run_surface(root: Path, metric_name: str, transform: str) -> dict:
    metric = resolve_drift_metric(metric_name, transform=transform)
    out_dir = LOCAL_OUT / root.name / output_subdir(metric, 'analysis')
    try:
        run_surface_analysis(str(root), str(out_dir), metric)
    except ValueError as exc:
        print(f'  {root.name} [{metric.key}]: skipped: {exc}')
        return {}
    path = out_dir / 'interaction_surface.json'
    return json.load(open(path)) if path.exists() else {}

In [ ]:
surface_results = {}
for root, info in experiment_roots.items():
    if info['kind'] != 'pout_sweep':
        continue
    for metric_name, transform in ANALYSES:
        key = (root.name, resolve_drift_metric(metric_name, transform=transform).key)
        surface_results[key] = s = run_surface(root, metric_name, transform)
        if s:
            table = pd.DataFrame({k: s[k] for k in (
                'alpha_values', 'beta_values', 'I_values', 'I_ci_lower', 'I_ci_upper',
                'p_values', 'n_units')})
            print(f"\n{key[0]} [{key[1]}]  R² = {s['r_squared']:.3f}")
            print(table.to_string(index=False))

## 6. Visualization

In [ ]:
from superadditivity.visualization.figure_style import set_publication_style
set_publication_style()

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

### 6a. Interaction bar charts (2×2 cells, additive prediction, I with CI)

In [ ]:
for root, info in experiment_roots.items():
    if info['kind'] != 'core_factorial':
        continue
    agg = ResultAggregator(str(root))
    for metric_name, transform in ANALYSES:
        metric = resolve_drift_metric(metric_name, transform=transform)
        png = LOCAL_OUT / root.name / f'interaction_bar_{metric.key}.png'
        # Reads the matching interaction_test.json written in section 4.
        analysis_root = LOCAL_OUT / root.name
        try:
            ok = plot_interaction_bar_figure(agg, metric, analysis_root, png)
        except ValueError as exc:
            print(f'{root.name} [{metric.key}]: {exc}')
            continue
        if ok:
            display(Markdown(f'**{root.name} [{metric.key}]**'))
            display(Image(str(png)))

### 6b. Interaction surface (pout sweeps)

In [ ]:
from superadditivity.visualization.interaction_surface import plot_interaction_surface

for (root_name, key), s in surface_results.items():
    if not s or len(s['I_values']) == 0:
        continue
    fig = plot_interaction_surface(
        np.array(s['alpha_values']), np.array(s['beta_values']), np.array(s['I_values']),
        title=f'{root_name}: I(α, p_out) [{key}]',
    )
    plt.show()

### 6c. Drift vs α across datasets (pout sweeps, drift orientation)

In [ ]:
metric = resolve_drift_metric('cka')
fig, ax = plt.subplots(figsize=(8, 5))
for root, info in experiment_roots.items():
    if info['kind'] != 'pout_sweep':
        continue
    df = ResultAggregator(str(root)).load_summaries()
    if metric.column not in df.columns:
        continue
    df = df[df[metric.column].notna()].copy()
    df['drift'] = metric.to_drift(df[metric.column])
    sub = df[df['p_out'] == df['p_out'].min()]      # strongest community structure
    noniid = sub[np.isfinite(sub['dirichlet_alpha'])].groupby('dirichlet_alpha')['drift'].mean()
    line, = ax.plot(noniid.index, noniid.values, 'o-', label=f'{root.name} (p_out={sub["p_out"].min():g})')
    iid = sub[np.isinf(sub['dirichlet_alpha'])]['drift'].mean()
    if np.isfinite(iid):
        ax.axhline(iid, color=line.get_color(), ls='--', lw=1)  # IID level
ax.set_xscale('log')
ax.set_xlabel('Dirichlet α (lower = communities more different); dashed = IID')
ax.set_ylabel(metric.label)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Summary Table

In [ ]:
rows = []
for (root_name, key), r in factorial_results.items():
    if r:
        rows.append({'experiment': root_name, 'metric': key, 'I': r['I_observed'],
                     'CI low': r['I_ci_lower'], 'CI high': r['I_ci_upper'],
                     'p (I<=0)': r['p_value'], 'seeds': r['n_units'],
                     'superadditive (p<0.05)': r['p_value'] < 0.05})
display(pd.DataFrame(rows))

for (root_name, key), s in surface_results.items():
    if s:
        I = np.array(s['I_values'])
        print(f"{root_name} [{key}]: R² = {s['r_squared']:.3f}, I > 0 at "
              f"{int((I > 0).sum())}/{len(I)} points, significant at "
              f"{int((np.array(s['p_values']) < 0.05).sum())}")